In [1]:
import time
from time import sleep
import numpy as np
from ms2k import MS2000
from TDI import TDI
from obis_laser_controller import ObisLaserController
from madpiezo import Madpiezo
import os
from RT_infer import RTInfer
from pathlib import Path
import imageio
import cv2

def main():
    # Initialize laser
    laser = ObisLaserController('COM6')
    laser.connect()
    laser.set_power( power = 0.066 ) # in Watts # max pawer is 0.066 W

    # Initialize TDI
    tdi = TDI()
    if not tdi.initialize_grabber():
        print("Failed to initialize TDI grabber. Exiting.")
        return
    
    # Load calibration image
    folder_path = "output/260424_yeast_actin/calibration"
    cal_profile = tdi.average_column_profile(folder_path, 20)
        
    # Initialize XY stage (MS2000 stage)
    print(f"COM Ports: {MS2000.scan_ports()}")
    ms2k = MS2000("COM5", 9600)
    ms2k.connect_to_serial()
    if not ms2k.is_open():
        print("Exiting the program...")
        return

    # Initialize Z stage (Madpiezo)
    piezo = Madpiezo()
    x0, y0, z0 = piezo.get_position()
    print(f"Initial piezo position: X={x0:.2f}, Y={y0:.2f}, Z={z0:.2f} µm")

    # Parameters
    x_speed = 0.074
    x_speed_return = 0.074 * 32
    y_speed = 0.074 * 32
    ms2k.set_max_speed("Y", y_speed)

    frame_num = 2
    x_move_distance = 80 * 10 * frame_num
    y_move_distance = -60 * 10
    row_num = 1
    num_repeats = 1
    time_interval = 0
    crop_info = {'crop_width': 1024, 'center_col': 2242}
    # crop_info = {'crop_width': 1024*2, 'center_col': 2242-1024} # Crop info WF (move y stage to -0.082 mm)


    # Z scanning parameters
    z_start = z0 - 0        # in µm
    z_end = z0 + 4         # in µm
    z_step = 0.15         # in µm
    z_positions = np.arange(z_start, z_end + z_step, z_step)

    # Output path
    output_path = "output/260424_yeast_actin/01_yeast_t1000_66mW"
    os.makedirs(output_path, exist_ok=True)
    print(f"Saving all images to: {output_path}")
        
    # Initialize RTInfer
    model_weights_dir = Path('C:/Users/Nikon_1/Documents/JupyterLab/hansol/ZS-DeconvNet/Python_MATLAB_Codes/your_saved_models/ellipses4000_with_100filaments_with_Gaussian0p5sigma_TDInoise3sigma_twostage_Unet_Hess0.02/weights_6000.h5')

    dummy_image = np.random.rand(1024, 1024).astype(np.float32) * 4095
    infer_runner = RTInfer(model_weights_path=model_weights_dir, raw_data=dummy_image)    
    
    # Store results
    all_frames_xyz = []
    initial_x = ms2k.get_position("X")
    initial_y = ms2k.get_position("Y")
    print(f"Initial XY position: X={initial_x} µm, Y={initial_y} µm")

    start_time = None
    capture_idx = 0

    # --- Main Scan Loop ---
    for i in range(num_repeats):
        print(f"\n=== Temporal Repeat {i + 1} of {num_repeats} ===")

        for z_idx, z_pos in enumerate(z_positions):
            print(f"\n>>> Moving to Z = {z_pos:.2f} µm (index {z_idx + 1}/{len(z_positions)})")
            piezo.mcl_write(z_pos, 3)  # Absolute move to Z
            sleep(0.05)  # allow piezo to settle

            for row_idx in range(row_num):
                print(f"\n--- Row {row_idx + 1} of {row_num} ---")

                laser.on()

                # Move X stage
                ms2k.set_max_speed("X", x_speed)
                ms2k.moverel_axis("X", x_move_distance)
                print(f"Started moving X stage to {x_move_distance} µm")

                if start_time is None:
                    start_time = time.time()

                frames = tdi.capture_frames_to_memory_live(
                    max_frames=frame_num,
                    display_zoom=0.5,
                    num_buffers=3,
                    cal_flag=1,
                    cal_profile=cal_profile,
                    crop_info=crop_info,
                    infer_flag=1,
                    infer_runner=infer_runner
                )

                timestamp_ms = int((time.time() - start_time) * 1000)
                print(f"Timestamp after capture: {timestamp_ms} ms")

                capture_idx += 1
                all_frames_xyz.append((i + 1, z_idx + 1, row_idx + 1, frames))

                laser.off()

                # Wait and return X
                ms2k.wait_for_device()
                ms2k.set_max_speed("X", x_speed_return)
                ms2k.moverel(-x_move_distance, 0)
                ms2k.wait_for_device()
                print("Returned X stage to original position")

                # Move Y to next row
                if row_idx < row_num - 1:
                    ms2k.moverel(0, y_move_distance)
                    ms2k.wait_for_device()
                    print(f"Moved Y stage by {y_move_distance} µm")

            # Return Y to initial after each Z plane
            ms2k.moverel(0, -y_move_distance * (row_num - 1))
            ms2k.wait_for_device()
            print("Returned Y stage to initial position")
        
        # Return to initial Z 
        piezo.mcl_write(z0, 3)
        
        time.sleep(time_interval)
    
    cv2.destroyAllWindows()

    # Save images
    for repeat_idx, z_idx, row_idx, frames in all_frames_xyz:
        subdir = os.path.join(output_path, f"t_{repeat_idx:03d}", f"z_{z_idx:03d}", f"row_{row_idx:03d}")
        os.makedirs(subdir, exist_ok=True)
        tdi.save_images_to_disk(frames, outdir=subdir, filename_prefix="frame", 
                                keys_to_save=['raw', 'calibrated', 'denoise', 'decon'])

    print(f"\nAll {capture_idx} captures completed and saved.")

    # Disconnect
    ms2k.disconnect_from_serial()
    piezo.mcl_close()
    laser.disconnect()


if __name__ == "__main__":
    main()


TensorFlow version: 2.5.0
Built with CUDA: True
GPU devices available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TF built with CUDA: 64_112
Interface:    PC3622 - Coaxlink Duo CXP-12 (1-camera, line-scan) - KDP01002
Device:       Device0
Resolution:   4640x1
Grabber successfully initialized.
COM Ports: ['COM4', 'COM5', 'COM6', 'COM7', 'COM9']
Connected to the serial port.
Serial port = COM5 :: Baud rate = 9600
Initial piezo position: X=-7.00, Y=-7.00, Z=3.93 µm
Send: SPEED Y=2.368
Recv: :N-1
Saving all images to: output/260424_yeast_actin/01_yeast_t1000_66mW
Initializing model...
Running dummy inference to warm up model...
[DEBUG] Raw image stats before normalization: min=0.008548356592655182, max=4094.999267578125, mean=2047.8067626953125
Input image shape before padding: (1024, 1024)
1/1 [==============================] - 17s 17s/step
Initialization complete.
Send: WHERE X
Recv: :A -716
Send: WHERE Y
Recv: :A -0
Initial XY position: X=-716 µm, Y=0 µm

=== Tem